# Module 5.7 — Speculative Decoding

**Goal:** Cut decoder latency 2–3× without changing the output distribution. A small draft model (Qwen2.5-0.5B) proposes γ tokens per step; the large target model (Qwen2.5-1.5B) verifies all γ in one parallel forward pass. Benchmark tokens/sec, acceptance rate α, and confirm output quality is unchanged.

## 1. Install & GPU Check

In [ ]:
# !pip install -q transformers accelerate rouge-score

import os, time, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 2. Configuration

In [ ]:
TARGET_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
DRAFT_MODEL  = 'Qwen/Qwen2.5-0.5B-Instruct'

MAX_NEW_TOKENS = 100
N_BENCH        = 20    # runs per latency measurement
GAMMA_DEFAULT  = 5     # draft tokens per step
GAMMA_SWEEP    = [2, 3, 5, 7, 10]
SMOKE_TEST     = True  # set False to load real models

os.makedirs('reports', exist_ok=True)

# Representative DeskMate tickets for benchmarking
BENCH_TICKETS = [
    'I was charged twice for my subscription last month.',
    'How do I reset my two-factor authentication device?',
    'The CSV export button is not working on my account.',
    'Can I get a refund after 30 days on the Pro plan?',
    'My iOS app keeps crashing after the latest update.',
]

GOLD = [
    {'ticket': 'I was charged twice for my subscription last month.',
     'ref': 'Contact support with your invoice numbers. We will investigate and refund within 3 business days.'},
    {'ticket': 'How do I reset my 2FA device?',
     'ref': 'Go to Account > Security > Two-Factor Authentication and click Reset Device.'},
    {'ticket': 'The CSV export button is not working.',
     'ref': 'This is a known issue ERR-500 fixed in version 4.3.0. Please update or contact support.'},
]
print('Config OK')

## 3. Load Target and Draft Models

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if SMOKE_TEST:
    print('SMOKE_TEST=True — skipping real model loads.')
    target_model = None
    draft_model  = None
    tokenizer    = None
else:
    print('Loading target model (1.5B)...')
    tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, token=HF_TOKEN)
    target_model = AutoModelForCausalLM.from_pretrained(
        TARGET_MODEL, torch_dtype=torch.float16, device_map='auto', token=HF_TOKEN)
    print('Target loaded.')

    print('Loading draft model (0.5B)...')
    draft_model = AutoModelForCausalLM.from_pretrained(
        DRAFT_MODEL, torch_dtype=torch.float16, device_map='auto', token=HF_TOKEN)
    print('Draft loaded.')

    vram = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
    print(f'VRAM used: {vram:.2f} GB')

## 4. Generation Helper

In [ ]:
def make_prompt(ticket, tok):
    if tok is None:
        return ticket
    msgs = [
        {'role': 'system', 'content': 'You are DeskMate, a concise support assistant.'},
        {'role': 'user',   'content': f'Ticket: {ticket}'},
    ]
    if hasattr(tok, 'apply_chat_template'):
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return f'Ticket: {ticket}\nAnswer:'

def generate(model, tok, ticket, assistant=None, n_tokens=MAX_NEW_TOKENS):
    if model is None:
        return '[simulated output]', []
    prompt = make_prompt(ticket, tok)
    inputs = tok(prompt, return_tensors='pt').to(DEVICE)
    kwargs = dict(max_new_tokens=n_tokens, do_sample=False)
    if assistant is not None:
        kwargs['assistant_model'] = assistant
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    new_ids = out[0][inputs['input_ids'].shape[1]:].tolist()
    text = tok.decode(new_ids, skip_special_tokens=True).strip()
    return text, new_ids

## 5. Standard Decoding Baseline

In [ ]:
def bench_tps(model, tok, tickets, assistant=None, n_runs=N_BENCH):
    if model is None:
        return None
    times = []
    for i in range(n_runs):
        ticket = tickets[i % len(tickets)]
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()
        generate(model, tok, ticket, assistant=assistant)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    median_sec = float(np.median(times))
    return round(MAX_NEW_TOKENS / median_sec, 1)

if SMOKE_TEST:
    print('SMOKE_TEST=True — using simulated throughput values.')
    std_tps  = 27.0
    spec_tps = 64.0
else:
    print('Benchmarking standard decoding...')
    std_tps = bench_tps(target_model, tokenizer, BENCH_TICKETS)

print(f'Standard decoding: {std_tps} tokens/sec')

## 6. Speculative Decoding Benchmark

In [ ]:
if not SMOKE_TEST:
    print('Benchmarking speculative decoding (γ=5)...')
    spec_tps = bench_tps(target_model, tokenizer, BENCH_TICKETS, assistant=draft_model)

speedup = round(spec_tps / std_tps, 2)
print(f'Speculative decoding: {spec_tps} tokens/sec')
print(f'Speedup:              {speedup}x')

## 7. Output Identity Check

Correct speculative decoding must produce **identical** outputs to standard decoding in greedy mode (`do_sample=False`). Any difference indicates an implementation bug.

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating output identity check.')
    print('[Simulated] All 3 outputs match: PASS')
    identity_pass = True
else:
    mismatches = 0
    for ex in GOLD:
        text_std,  ids_std  = generate(target_model, tokenizer, ex['ticket'])
        text_spec, ids_spec = generate(target_model, tokenizer, ex['ticket'],
                                       assistant=draft_model)
        match = ids_std == ids_spec
        print(f'  Ticket: {ex["ticket"][:40]}...')
        print(f'  Match: {match}')
        if not match:
            mismatches += 1
            print(f'  STD:  {text_std[:80]}')
            print(f'  SPEC: {text_spec[:80]}')
    identity_pass = mismatches == 0
    print(f'\nIdentity gate: {"PASS" if identity_pass else f"FAIL ({mismatches} mismatches)"}')

## 8. Acceptance Rate α Estimation

In [ ]:
# α = fraction of draft tokens accepted per speculative step.
# In practice we estimate it from the ratio of spec_tps to std_tps:
# speedup ≈ (1 + γ·α) / (1 + γ·C_ratio)
# Rearranging for α (assuming C_ratio ≈ 0.15 for 0.5B vs 1.5B):

C_RATIO = 0.15   # approximate cost(draft) / cost(target)
GAMMA   = GAMMA_DEFAULT

alpha_est = (speedup * (1 + GAMMA * C_RATIO) - 1) / GAMMA
alpha_est = round(max(0, min(1, alpha_est)), 3)

print(f'Estimated acceptance rate α ≈ {alpha_est}')
print(f'(based on speedup={speedup}x, γ={GAMMA}, C_ratio={C_RATIO})')
print()
print('Interpretation:')
if alpha_est >= 0.75:
    print(f'  α={alpha_est} is high — draft model distribution closely matches target.')
    print('  Consider fine-tuning draft on DeskMate data to push α toward 0.85.')
elif alpha_est >= 0.55:
    print(f'  α={alpha_est} is moderate — reasonable for a generic draft model.')
    print('  Fine-tuning draft on DeskMate data should improve acceptance.')
else:
    print(f'  α={alpha_est} is low — draft and target distributions diverge significantly.')
    print('  Consider: same-family draft, lower temperature, or domain fine-tuning.')

## 9. Draft Length Sweep — Find Optimal γ

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating γ sweep.')
    # Simulated: peak speedup around γ=5
    gamma_tps = {2: 42.0, 3: 54.0, 5: 64.0, 7: 61.0, 10: 55.0}
else:
    gamma_tps = {}
    for g in GAMMA_SWEEP:
        # Temporarily patch num_assistant_tokens by checking HF API
        # HF transformers >= 4.38 accepts num_assistant_tokens in GenerationConfig
        tps = bench_tps(target_model, tokenizer, BENCH_TICKETS[:3],
                        assistant=draft_model, n_runs=10)
        gamma_tps[g] = tps
        print(f'  γ={g}: {tps} tokens/sec')

best_gamma = max(gamma_tps, key=gamma_tps.get)
best_gamma_tps = gamma_tps[best_gamma]
print(f'\nBest γ: {best_gamma} at {best_gamma_tps} tokens/sec')

## 10. ROUGE-L Quality Check

In [ ]:
from rouge_score import rouge_scorer as rs_module

_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulated ROUGE-L values.')
    std_rouge  = 0.470
    spec_rouge = 0.470
    delta = 0.000
else:
    std_scores  = []
    spec_scores = []
    for ex in GOLD:
        t_std,  _ = generate(target_model, tokenizer, ex['ticket'])
        t_spec, _ = generate(target_model, tokenizer, ex['ticket'], assistant=draft_model)
        std_scores.append(_scorer.score(ex['ref'], t_std)['rougeL'].fmeasure)
        spec_scores.append(_scorer.score(ex['ref'], t_spec)['rougeL'].fmeasure)
    std_rouge  = round(sum(std_scores)  / len(std_scores),  4)
    spec_rouge = round(sum(spec_scores) / len(spec_scores), 4)
    delta = round(spec_rouge - std_rouge, 4)

print(f'Standard decoding ROUGE-L:   {std_rouge}')
print(f'Speculative decoding ROUGE-L: {spec_rouge}')
print(f'Delta: {delta}')
quality_gate = abs(delta) <= 0.001
print(f'Quality gate (|delta| <= 0.001): {"PASS" if quality_gate else "REVIEW"}')
print('(Identical greedy outputs → delta should be exactly 0.0)')

## 11. Results Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Throughput comparison
ax = axes[0]
ax.bar(['Standard', 'Speculative'], [std_tps, spec_tps],
       color=['steelblue', 'coral'])
ax.set_ylabel('Tokens / second')
ax.set_title(f'Throughput ({speedup}x speedup)')
for i, v in enumerate([std_tps, spec_tps]):
    ax.text(i, v + 0.5, str(v), ha='center', fontsize=10)

# γ sweep
ax = axes[1]
gammas = sorted(gamma_tps)
tps_vals = [gamma_tps[g] for g in gammas]
ax.plot(gammas, tps_vals, marker='o', color='seagreen')
ax.axvline(best_gamma, color='coral', linestyle='--', label=f'Best γ={best_gamma}')
ax.set_xlabel('Draft length γ')
ax.set_ylabel('Tokens / second')
ax.set_title('Throughput vs Draft Length γ')
ax.legend()

# Acceptance rate illustration
ax = axes[2]
alpha_vals = [max(0, (gamma_tps[g] / std_tps * (1 + g*C_RATIO) - 1) / g)
              for g in gammas]
ax.plot(gammas, [round(a, 3) for a in alpha_vals], marker='s', color='darkorange')
ax.axhline(alpha_est, color='steelblue', linestyle='--', label=f'α≈{alpha_est}')
ax.set_xlabel('Draft length γ')
ax.set_ylabel('Estimated acceptance rate α')
ax.set_title('Acceptance Rate vs γ')
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.savefig('speculative_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: speculative_results.png')

## 12. Checkpoint: Why Outputs Are Statistically Identical

In [ ]:
checkpoint = '''
Why does correct speculative decoding produce outputs statistically identical
to standard target-only decoding?

The acceptance criterion -- accept draft token x~ with probability
    min(1, p(x~) / q(x~))
-- is exactly the rejection sampling rule for drawing from distribution p
using proposal distribution q.

Each accepted token's marginal distribution is p (the target), not q (the draft).
When a token is rejected at position k, it is resampled from:
    p'(x) = max(0, p(x) - q(x)) / Z
which is the residual of p not covered by q -- also a distribution with marginal p.

Since every token in the final output is drawn from p (via acceptance or correction),
the output sequence has the exact same distribution as standard target-only decoding.
Speculative decoding is a provably lossless speedup -- not an approximation.
'''
print(checkpoint)

## 13. Save Report

In [ ]:
report = [
    '# Speculative Decoding Report\n',
    f'Target model: {TARGET_MODEL}',
    f'Draft model:  {DRAFT_MODEL}',
    f'Smoke test:   {SMOKE_TEST}\n',
    '## Results',
    '',
    '| Metric | Value |',
    '|--------|-------|',
    f'| Standard decoding (tokens/sec) | {std_tps} |',
    f'| Speculative decoding (tokens/sec) | {spec_tps} |',
    f'| Speedup | {speedup}x |',
    f'| Acceptance rate α (estimated) | {alpha_est} |',
    f'| Best draft length γ | {best_gamma} |',
    f'| ROUGE-L standard | {std_rouge} |',
    f'| ROUGE-L speculative | {spec_rouge} |',
    f'| ROUGE-L delta | {delta} |',
    f'| Output identity | {"PASS" if identity_pass else "FAIL"} |',
    '',
    '## Checkpoint',
    '',
    'Speculative decoding is statistically identical to standard decoding because',
    'the acceptance criterion min(1, p/q) is exactly rejection sampling from p.',
    'Rejected tokens are resampled from the corrected residual distribution p\' = max(0,p-q)/Z,',
    'which also has marginal p. Every output token is drawn from p -- provably lossless.',
]

with open('reports/speculative_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/speculative_report.md')
print('\n=== Module 5.7 Complete ===')
print(f'Throughput: {std_tps} -> {spec_tps} tokens/sec  ({speedup}x speedup)')
print(f'Acceptance rate α ≈ {alpha_est}  |  Best γ = {best_gamma}')
print(f'Output identity: {"PASS" if identity_pass else "FAIL"}')